# PhishGuard AI — Kaggle-Only Training Notebook

This notebook rebuilds the **same cleaning and training flow** used in the PhishGuard AI notebook, but uses **only the Kaggle email dataset**.

## What this notebook does
- Mounts Google Drive
- Loads Kaggle credentials
- Downloads the Kaggle phishing email dataset
- Applies the **same refined cleaning logic** used in PhishGuard AI
- Keeps only the Kaggle dataset (no combined dataset)
- Splits train/test properly
- Rebalances the **training set only**
- Trains a TF-IDF + Logistic Regression phishing classifier
- Evaluates the model
- Saves:
  - cleaned Kaggle dataset
  - TF-IDF vectorizer
  - trained model

## Dataset
Kaggle dataset used here:
`subhajournal/phishingemails`

In [ ]:
# ============================================================
# 1. Core imports
# ============================================================

import os
import re
import joblib
import warnings
import zipfile
import subprocess
import sys

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

warnings.filterwarnings("ignore")

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)

In [ ]:
# ============================================================
# 2. Mount Google Drive
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

## Kaggle credential setup

This notebook first tries to use:

`/content/drive/MyDrive/kaggle.json`

If that file does not exist, it will ask you to upload `kaggle.json`.

In [ ]:
# ============================================================
# 3. Kaggle API setup
# ============================================================

from google.colab import files

KAGGLE_DIR = os.path.expanduser("~/.kaggle")
KAGGLE_JSON_LOCAL = os.path.join(KAGGLE_DIR, "kaggle.json")
KAGGLE_JSON_DRIVE = "/content/drive/MyDrive/kaggle.json"

os.makedirs(KAGGLE_DIR, exist_ok=True)

if os.path.exists(KAGGLE_JSON_DRIVE):
    !cp "/content/drive/MyDrive/kaggle.json" ~/.kaggle/kaggle.json
    print("Using kaggle.json from Google Drive.")
else:
    print("kaggle.json not found in Google Drive.")
    print("Please upload your kaggle.json file now.")
    uploaded = files.upload()
    if "kaggle.json" not in uploaded:
        raise FileNotFoundError("kaggle.json was not uploaded.")
    !cp kaggle.json ~/.kaggle/kaggle.json
    print("Uploaded kaggle.json successfully.")

!chmod 600 ~/.kaggle/kaggle.json

try:
    import kaggle
except ImportError:
    !pip -q install kaggle
    import kaggle

print("Kaggle API is ready.")

In [ ]:
# ============================================================
# 4. Download the Kaggle dataset
# ============================================================

DATASET_ID = "subhajournal/phishingemails"
DOWNLOAD_DIR = "/content/kaggle_email_data"

os.makedirs(DOWNLOAD_DIR, exist_ok=True)

!kaggle datasets download -d $DATASET_ID -p $DOWNLOAD_DIR --force

zip_files = [f for f in os.listdir(DOWNLOAD_DIR) if f.endswith(".zip")]
if not zip_files:
    raise FileNotFoundError("No zip file downloaded from Kaggle.")

zip_path = os.path.join(DOWNLOAD_DIR, zip_files[0])

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(DOWNLOAD_DIR)

print("Downloaded and extracted files:")
for root, dirs, files_ in os.walk(DOWNLOAD_DIR):
    for f in files_:
        print(os.path.join(root, f))

In [ ]:
# ============================================================
# 5. Load the Kaggle dataset only
# ============================================================

candidate_csvs = []
for root, dirs, files_ in os.walk(DOWNLOAD_DIR):
    for f in files_:
        if f.lower().endswith(".csv"):
            candidate_csvs.append(os.path.join(root, f))

print("CSV candidates found:")
for p in candidate_csvs:
    print(" -", p)

if not candidate_csvs:
    raise FileNotFoundError("No CSV file found in extracted Kaggle dataset.")

# Prefer the known file name if present
preferred = [p for p in candidate_csvs if os.path.basename(p).lower() == "phishing_email.csv"]
csv_path = preferred[0] if preferred else candidate_csvs[0]

print("\nUsing CSV:", csv_path)

df_raw = pd.read_csv(csv_path)

print("\nRaw shape:", df_raw.shape)
print("\nColumns:")
print(df_raw.columns.tolist())
display(df_raw.head())

## Standardize columns to match the PhishGuard AI training notebook

In [ ]:
# ============================================================
# 6. Standardize schema for Kaggle dataset
# ============================================================

df = df_raw.copy()

# Common expected columns in this dataset:
# - "Email Text"
# - "Email Type"

if "Email Text" not in df.columns:
    raise ValueError("Expected column 'Email Text' not found.")

if "Email Type" not in df.columns:
    raise ValueError("Expected column 'Email Type' not found.")

df = df.rename(columns={"Email Text": "text"})

df["label"] = df["Email Type"].map({
    "Phishing Email": 1,
    "Safe Email": 0
})

print("Missing values before cleanup:")
print(df[["text", "label"]].isna().sum())

invalid_label_rows = df["label"].isna().sum()
print("\nRows with unmapped labels:", invalid_label_rows)

df = df.dropna(subset=["text", "label"]).copy()
df["text"] = df["text"].astype(str)
df["label"] = df["label"].astype(int)

print("\nUnique labels:")
print(sorted(df["label"].unique().tolist()))

display(df.head())

## Apply the same refined cleaning logic from PhishGuard AI

In [ ]:
# ============================================================
# 7. Refined cleaning logic from PhishGuard AI
# ============================================================

def clean_email_text(raw_text: str) -> str:
    if not isinstance(raw_text, str):
        return ""

    text = raw_text.replace("\r\n", "\n").replace("\r", "\n").strip()

    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Normalize whitespace early
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n", text).strip()

    # If this looks like a raw Enron-style flattened header email,
    # remove everything up to the end of the top metadata block.
    if text.lower().startswith("message-id:"):
        subject_match = re.search(
            r"subject:\s*(.*?)\s+(?:mime-version:|content-type:|content-transfer-encoding:|x-from:|x-to:|x-cc:|x-bcc:|x-folder:|x-origin:|x-filename:)",
            text,
            flags=re.IGNORECASE | re.DOTALL
        )
        subject = subject_match.group(1).strip() if subject_match else ""

        text = re.sub(
            r"^.*?x-filename:\s*.*?(?=(?:-{5,}\s*forwarded by)|(?:enron capital & trade resources corp\.)|(?:from:\s*\"?[A-Za-z])|(?:[0-9]{1,2}:[0-9]{2}\s*gmt)|(?:[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\s*\(Dow Jones\))|(?:[A-Za-z].{20,}))",
            "",
            text,
            flags=re.IGNORECASE | re.DOTALL
        )

        if len(text.strip()) == 0 or text.lower().startswith("message-id:"):
            fallback = re.split(r"x-filename:\s*.*?", text, flags=re.IGNORECASE, maxsplit=1)
            if len(fallback) == 2:
                text = fallback[1].strip()

        text = re.sub(r"-{5,}\s*forwarded by.*?-{5,}", " ", text, flags=re.IGNORECASE | re.DOTALL)

        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\n\s*\n+", "\n", text).strip()

        if subject and not text.lower().startswith("subject:"):
            text = f"Subject: {subject}\n{text}"

    else:
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\n\s*\n+", "\n", text).strip()

    return text.strip()

df["text"] = df["text"].apply(clean_email_text)

empty_text_count = (df["text"].str.strip() == "").sum()
print("Empty text rows after cleaning:", empty_text_count)

df = df[df["text"].str.strip() != ""].copy()

duplicate_count = df.duplicated().sum()
print("Duplicate rows before removal after cleaning:", duplicate_count)

df = df.drop_duplicates().copy().reset_index(drop=True)

print("\nFinal shape after cleaning:", df.shape)
print("\nClass distribution:")
print(df["label"].value_counts())

print("\nSample cleaned rows:")
display(df.head(5))

In [ ]:
# ============================================================
# 8. Save the cleaned Kaggle-only dataset to Google Drive
# ============================================================

SAVE_DIR = "/content/drive/MyDrive/fyp_kaggle_only"
os.makedirs(SAVE_DIR, exist_ok=True)

cleaned_dataset_path = os.path.join(SAVE_DIR, "kaggle_only_cleaned_email_dataset.csv")
df.to_csv(cleaned_dataset_path, index=False)

print("Saved cleaned dataset to:")
print(cleaned_dataset_path)

## Train/test split
This happens **before TF-IDF** so the vectorizer learns only from the training set.

In [ ]:
# ============================================================
# 9. Define features and labels
# ============================================================

X = df["text"].astype(str).values
y = df["label"].astype(int).values

print("Total samples:", len(X))

In [ ]:
# ============================================================
# 10. Train-test split
# ============================================================

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train_text))
print("Testing samples :", len(X_test_text))

## Rebalance the training set only

Same strategy as the PhishGuard AI notebook:
- keep the test set untouched
- keep all phishing emails in training
- sample legitimate emails to at most 2x phishing count

In [ ]:
# ============================================================
# 11. Rebalance training set only
# ============================================================

train_df = pd.DataFrame({
    "text": X_train_text,
    "label": y_train
})

train_legit = train_df[train_df["label"] == 0].copy()
train_phish = train_df[train_df["label"] == 1].copy()

print("Original training set class distribution:")
print(train_df["label"].value_counts())

phish_count = len(train_phish)
target_legit_count = min(len(train_legit), phish_count * 2)

train_legit_sampled = train_legit.sample(
    n=target_legit_count,
    random_state=42
)

train_balanced = pd.concat([train_legit_sampled, train_phish], axis=0)
train_balanced = train_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

X_train_text = train_balanced["text"].values
y_train = train_balanced["label"].values

print("\nRebalanced training set class distribution:")
print(pd.Series(y_train).value_counts())

## TF-IDF feature extraction
Same configuration as PhishGuard AI.

In [ ]:
# ============================================================
# 12. TF-IDF vectorization
# ============================================================

tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

X_train = tfidf_vectorizer.fit_transform(X_train_text)
X_test = tfidf_vectorizer.transform(X_test_text)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

feature_names = tfidf_vectorizer.get_feature_names_out()
print("\nFirst 20 TF-IDF features:")
print(feature_names[:20])

## Train classifier
Same model family as PhishGuard AI.

In [ ]:
# ============================================================
# 13. Train Logistic Regression model
# ============================================================

phishing_model = LogisticRegression(
    max_iter=2000,
    random_state=42
)

phishing_model.fit(X_train, y_train)

print("Model training complete.")

## Evaluate model

In [ ]:
# ============================================================
# 14. Predictions
# ============================================================

y_pred = phishing_model.predict(X_test)
y_proba = phishing_model.predict_proba(X_test)[:, 1]

In [ ]:
# ============================================================
# 15. Metrics
# ============================================================

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

In [ ]:
# ============================================================
# 16. Confusion matrix
# ============================================================

cm = confusion_matrix(y_test, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual Legitimate", "Actual Phishing"],
    columns=["Predicted Legitimate", "Predicted Phishing"]
)

display(cm_df)

In [ ]:
# ============================================================
# 17. Classification report
# ============================================================

print(classification_report(y_test, y_pred, target_names=["Legitimate", "Phishing"]))

In [ ]:
# ============================================================
# 18. Inspect sample predictions
# ============================================================

sample_count = min(5, len(X_test_text))

for i in range(sample_count):
    snippet = X_test_text[i][:250].replace("\n", " ")
    true_label = y_test[i]
    pred_label = y_pred[i]
    probability = y_proba[i]

    print(f"Sample {i}")
    print(f"Text snippet        : {snippet}...")
    print(f"True label          : {true_label}")
    print(f"Predicted label     : {pred_label}")
    print(f"Phishing probability: {probability:.4f}")
    print("-" * 100)

## Save artifacts to Google Drive

In [ ]:
# ============================================================
# 19. Save model artifacts
# ============================================================

vectorizer_path = os.path.join(SAVE_DIR, "tfidf_vectorizer_kaggle_only.pkl")
model_path = os.path.join(SAVE_DIR, "phishing_model_kaggle_only.pkl")

joblib.dump(tfidf_vectorizer, vectorizer_path)
joblib.dump(phishing_model, model_path)

print("Saved vectorizer to:", vectorizer_path)
print("Saved model to     :", model_path)

In [ ]:
# ============================================================
# 20. Final summary
# ============================================================

print("Done.")
print("\nArtifacts saved in:")
print(SAVE_DIR)
print("\nFiles expected:")
print(" - kaggle_only_cleaned_email_dataset.csv")
print(" - tfidf_vectorizer_kaggle_only.pkl")
print(" - phishing_model_kaggle_only.pkl")